# Phase 1 — Data Cleaning & Feature Engineering

Notebook 01 of the *Amazon Product Intelligence* project.

This notebook walks through, and validates, the cleaning + feature-engineering
pipeline implemented in `src/preprocessing.py` and `src/feature_engineering.py`.
The notebook calls into those modules rather than duplicating logic, so the
pipeline stays defined in exactly one place (importable by the dashboard,
other notebooks, and tests alike).

**Steps covered**
1. Load dataset & overview (shape, dtypes, missing values, duplicates)
2. Convert price columns (₹1,099 → 1099.0)
3. Convert rating_count (24,269 → 24269)
4. Category processing → main_category / sub_category / product_type
5. Text cleaning (about_product, review_title, review_content)
6. Feature engineering (discount_amount, *_length, popularity_score, buckets)
7. Save `data/processed/cleaned_products.csv` and `products_features.csv`


In [ ]:
import sys
from pathlib import Path

# Make the repo root importable so `import src...` works when running from notebooks/
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## 1. Load Dataset

In [ ]:
from src.utils import raw_data_path

df = pd.read_csv(raw_data_path("amazon.csv"))
df.shape


In [ ]:
checks = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isnull().sum(),
    "missing_pct": (df.isnull().mean() * 100).round(2),
})
checks


In [ ]:
print("Duplicate rows (all columns):", df.duplicated().sum())
print("Duplicate product_id:", df.duplicated(subset=['product_id']).sum())


There are duplicate `product_id` values in the raw file (repeat listings scraped
more than once). These are dropped in the cleaning pipeline, keeping the first
occurrence, so downstream product-level stats aren't double-counted.

## 2–5. Run the Cleaning Pipeline (`src/preprocessing.py`)

In [ ]:
from src.preprocessing import (
    load_raw_data, drop_exact_duplicates, clean_price_columns,
    clean_rating_columns, clean_discount_percentage, split_category,
    clean_text_columns, run_cleaning_pipeline,
)

# Step-by-step, so we can inspect each transformation individually
raw = load_raw_data()
step = drop_exact_duplicates(raw)
step = clean_price_columns(step)
step = clean_rating_columns(step)
step = clean_discount_percentage(step)
step = split_category(step)
step = clean_text_columns(step)

step[["discounted_price", "actual_price", "rating", "rating_count",
      "discount_percentage"]].describe()


In [ ]:
# Category split check
step[["category", "main_category", "sub_category", "product_type"]].drop_duplicates(
    subset=["main_category", "sub_category", "product_type"]
).head(15)


In [ ]:
# Text cleaning check — before / after for one row
row = step.iloc[0]
print("BEFORE:\n", row["about_product"][:300])
print()
print("AFTER:\n", row["about_product_clean"][:300])


In [ ]:
# Or just run the whole pipeline in one call (this is what src/preprocessing.py
# does when run directly with `python src/preprocessing.py`) — saves to
# data/processed/cleaned_products.csv
cleaned_df = run_cleaning_pipeline()
cleaned_df.shape


## 6. Feature Engineering (`src/feature_engineering.py`)

In [ ]:
from src.feature_engineering import run_feature_engineering

features_df = run_feature_engineering(cleaned_df)
features_df[[
    "discount_amount", "review_length", "title_length", "description_length",
    "popularity_score", "price_bucket", "discount_bucket",
]].describe(include="all")


In [ ]:
features_df["price_bucket"].value_counts()


In [ ]:
features_df["discount_bucket"].value_counts()


## 7. Outputs

- `data/processed/cleaned_products.csv` — cleaned prices/ratings/category/text, one row per product
- `data/processed/products_features.csv` — the above plus engineered features

Both files are written automatically by the pipeline calls above and are consumed
by the next notebook, `02_eda.ipynb` (Phase 2).

In [ ]:
features_df.to_csv("../data/processed/products_features.csv", index=False)
print("Saved:", features_df.shape)
